In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, VBox, HBox, HTML, interactive_output
from IPython.display import display

# Configuration
N = 64  # Length of the window / DFT size

def plot_leakage_demo(f_index):
    plt.close('all')
    
    # Time indices
    n = np.arange(N)
    
    # Input signal: Cosine with frequency index f_index
    x = np.cos(2 * np.pi * f_index * n / N)
    
    # 1. Compute standard N-point DFT
    X_dft = np.fft.fft(x)
    freqs_dft = np.arange(N)
    
    # 2. Compute high-resolution continuous spectrum (DTFT)
    n_fft = 2048
    x_padded = np.zeros(n_fft)
    x_padded[:N] = x
    X_dtft = np.fft.fft(x_padded, n_fft)
    freqs_dtft = np.fft.fftfreq(n_fft, d=1/N)
    
    # Check if f_index is very close to an integer (no leakage condition)
    is_integer = np.isclose(f_index, np.round(f_index), atol=1e-3)
    
    # Figure layout with a GridSpec to leave room or side info if needed, 
    # but here we use a side-by-side widget layout (matplotlib + HTML panel)
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Plot continuous spectrum (DTFT magnitude in positive frequencies)
    pos_mask = (freqs_dtft >= 0) & (freqs_dtft <= N/2)
    ax.plot(freqs_dtft[pos_mask], np.abs(X_dtft[pos_mask]) / 2, 'r-', lw=1.5, label='Continuous Spectrum (DTFT)')
    
    # Plot discrete DFT samples as stems
    dft_mask = freqs_dft <= N/2
    ax.stem(freqs_dft[dft_mask], np.abs(X_dft[dft_mask]) / 2, 
            linefmt='C0-', markerfmt='C0o', basefmt='k-', label='DFT Samples ($X(k)$)')
    
    status_text = "NO LEAKAGE (Matched Grid)" if is_integer else "LEAKAGE PRESENT (Off-Grid)"
    status_color = "green" if is_integer else "red"
    
    ax.set_title(f"Frequency Index $m = {f_index:.2f}$ -> {status_text}", fontsize=10, fontweight='bold', color=status_color)
    ax.set_xlabel("Frequency Index $k$", fontsize=9)
    ax.set_ylabel(r"$|\mathcal{Y}(e^{j\omega_k})|^+$", fontsize=9)
    ax.set_xlim(0, N/2)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend(loc='upper right', fontsize=8)
    
    plt.tight_layout()
    plt.show()

# Create interactive output for the plot
ui_plot = interactive_output(plot_leakage_demo, {'f_index': FloatSlider(value=4.25, min=0, max=16, step=0.05)})

# Create a side panel (HTML) showing the valid DFT frequency indices and instructions
info_html = HTML("""
<div style="padding: 15px; background-color: #f8f9fa; border: 1px solid #ddd; border-radius: 5px; font-family: sans-serif; font-size: 13px; width: 260px;">
    <h4 style="margin-top: 0; color: #333;">💡 DFT Frequencies Guide</h4>
    <p>The discrete frequencies of the DFT are located at integer index values:</p>
    <p style="font-family: monospace; background: #eee; padding: 5px; text-align: center; font-weight: bold;">
        <i>k</i> = 0, 1, 2, 3, ..., 32
    </p>
    <p><b>Rule for Zero Leakage:</b></p>
    <p>Set the slider to an <b>exact integer</b> (e.g., <b>4, 8, 12, 16</b>). When <i>m</i> is an integer, the signal frequency matches the DFT grid, and leakage vanishes!</p>
    <p><b>Current Status:</b> Move the slider to see the change.</p>
</div>
""")

# Slider definition
freq_slider = FloatSlider(value=4.25, min=0, max=16, step=0.5, description='Index (m):', style={'description_width': '70px'})

# Re-link slider properly to both the plot function and layout
out_plot = interactive_output(plot_leakage_demo, {'f_index': freq_slider})

# Arrange layout side-by-side: Left side is slider+plot, Right side is the info panel
dashboard = HBox([
    VBox([freq_slider, out_plot]),
    info_html
])

display(dashboard)